<a href="https://colab.research.google.com/github/alemaria165-star/CodeAlexiaR/blob/main/robotcar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pybullet
!git clone https://github.com/lukeje/robot-example.git
!pip install ./robot-example

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# adapted from https://github.com/akinami3/PybulletRobotics

import math
import time

from importlib.util import find_spec

import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib import rc
rc('animation', html='jshtml')

import numpy as np
import pybullet as pb
import pybullet_data
from robot_example import controlcar


In [ ]:
# Connect to the display
pb.connect(pb.DIRECT)

In [ ]:
# Set simulation parameters
pb.resetSimulation() # Reset the simulation space
pb.setAdditionalSearchPath(pybullet_data.getDataPath()) # Add path to necessary data for pybullet
pb.setGravity(0.0, 0.0, -9.8) # Set gravity as on Earth
timestep = 1.0/240.0
pb.setTimeStep(timestep) # Set the elapsed time per step
nsteps = 500 # How many steps taken for each motion

# Load the floor
plane_id = pb.loadURDF("plane.urdf")

# Load the robot
car_start_pos = [0, 0, 0.1] # Set the initial position (x, y, z)
car_start_orientation = pb.getQuaternionFromEuler([0, 0, 0]) # Set the initial orientation (roll, pitch, yaw)
car_id = pb.loadURDF("robot-example/data/PybulletRobotics/urdf/simple_two_wheel_car.urdf", car_start_pos, car_start_orientation)

In [ ]:
# Set up camera
pb.configureDebugVisualizer(pb.COV_ENABLE_RENDERING, 0)
pb.configureDebugVisualizer(pb.COV_ENABLE_GUI, 0)
camera_target_position = [0.0, 0.0, 0.0]
viewMatrix = pb.computeViewMatrixFromYawPitchRoll(camera_target_position, 2, 0, -20, 0, 2)

In [ ]:
# Indices for joints
RIGHT_WHEEL_JOINT_IDX = 0
LEFT_WHEEL_JOINT_IDX  = 1

# Instantiate class to define the wheel velocities
mv = controlcar.MovementVelocities(timestep,nsteps)

# Loop over wheel velocities for different states
fig, ax = plt.subplots()
artists = []
frame = 0
for v in (mv.goforwards(1), mv.gobackwards(0.5), mv.turnleft(math.pi/2), mv.turnright(math.pi)):
    # Set the target velocities for each wheel
    pb.setJointMotorControl2(car_id, RIGHT_WHEEL_JOINT_IDX, pb.VELOCITY_CONTROL, targetVelocity=v[RIGHT_WHEEL_JOINT_IDX])
    pb.setJointMotorControl2(car_id, LEFT_WHEEL_JOINT_IDX,  pb.VELOCITY_CONTROL, targetVelocity=v[LEFT_WHEEL_JOINT_IDX])

    # Step the simulation forwards in time
    for _ in range(nsteps):
        pb.stepSimulation()

        # capture only every 20th frame
        if not (frame % 20):
            img_arr = pb.getCameraImage(240, 180,
                                    viewMatrix=viewMatrix,
                                    shadow=1, lightDirection=[1, 1, 1],
                                    renderer=pb.ER_BULLET_HARDWARE_OPENGL)

            w = img_arr[0]  # width of the image, in pixels
            h = img_arr[1]  # height of the image, in pixels
            rgb = img_arr[2]  # color data RGB
            dep = img_arr[3]  # depth data

            # reshape is not needed
            np_img_arr = np.reshape(rgb, (h, w, 4))
            np_img_arr = np_img_arr * (1. / 255.)

            # note that sending the data to matplotlib is really slow
            im = ax.imshow(np_img_arr, animated=True)
            artists.append([im])
        frame += 1

In [ ]:
ani = animation.ArtistAnimation(fig=fig, artists=artists, interval=timestep, blit=True)
ani